# Preparação de Dados - Imóveis de São Paulo

## Propósito

Este script Python realiza a preparação e processamento de dados imobiliários da cidade de São Paulo, especificamente dados de IPTU (Imposto Predial e Territorial Urbano) e ITBI (Imposto sobre Transmissão de Bens Imóveis) para o período de 2019 a 2024. O objetivo principal é calcular o **Valor Venal** dos imóveis utilizando fórmulas complexas que consideram diversos fatores como área construída, área do terreno, profundidade, tipo de terreno, localização (Zona Fiscal), entre outros.

## Entradas (Inputs)

### Dados de IPTU
- **Arquivos Excel**: Múltiplos arquivos Excel contendo dados de IPTU com as seguintes colunas:
  - Número do Contribuinte
  - Ano do Exercício
  - Nome de Logradouro do Imóvel
  - Número do Imóvel
  - Área do Terreno
  - Área Construída
  - Área Ocupada
  - Valor do m² do Terreno
  - Valor do m² de Construção
  - Tipo de Uso do Imóvel
  - Tipo de Padrão da Construção
  - Testada para Cálculo
  - Fração Ideal
  - Fator de Obsolescência
  - Tipo de Terreno
  - Quantidade de Esquinas Frentes

### Dados de ITBI
- **Arquivos Excel**: Arquivos nomeados como `ITBI_2019.xlsx` até `ITBI_2024.xlsx`, cada um contendo planilhas mensais (JAN-YYYY, FEV-YYYY, etc.) com dados de transações imobiliárias.

### Dicionários de Mapeamento
- **fator_prof_map**: Mapeamento de profundidade para fatores de correção
- **fator_terreno**: Fatores para diferentes tipos de terreno (lote de fundos, terreno interno)
- **fator_ZF**: Fatores por Zona Fiscal (1-4)
- **fator_ea**: Fatores de excesso de área por Zona Fiscal
- **area_map**: Mapeamento de áreas por quantidade de esquinas
- **zf_map**: Mapeamento de códigos de contribuinte para Zonas Fiscais

## Saídas (Outputs)

### Arquivo CSV - IPTU
- **Nome**: `data/IPTU_2019_2024.csv`
- **Conteúdo**: Dataset processado contendo:
  - Todas as colunas originais selecionadas
  - Coluna **VALOR VENAL** calculada através da função `calcula_vv()`, que considera:
    - Valor da construção (área construída × valor m² × fator de obsolescência)
    - Valor do terreno incorporado (com ajustes por profundidade, tipo de terreno, esquina)
    - Valor do excesso de área do terreno
    - Fatores de condomínio para imóveis verticais
    - Fração ideal do imóvel

### Arquivo CSV - ITBI
- **Nome**: `ITBI_2019_2024.csv`
- **Conteúdo**: Dataset consolidado de transações imobiliárias contendo:
  - Número do Cadastro (SQL)
  - Natureza de Transação
  - Valor de Transação (declarado pelo contribuinte)
  - Data de Transação
  - Valor Venal de Referência
  - Proporção Transmitida (%)
  - Valor Venal de Referência (proporcional)
  - Base de Cálculo adotada
  - Tipo de Financiamento
  - Valor Financiado
  - Situação do SQL
  - Ano, Mês e Ano/Mês da Transação

## Processamento Realizado

1. **Carregamento e Consolidação**: Combina múltiplos arquivos de IPTU e ITBI em dataframes únicos
2. **Limpeza de Dados**: Remove registros inválidos, normaliza tipos de dados e formata campos
3. **Cálculo de Zona Fiscal**: Extrai a Zona Fiscal (ZF) a partir dos primeiros 3 dígitos do número do contribuinte
4. **Cálculo do Valor Venal**: Aplica fórmula complexa que considera múltiplos fatores de correção
5. **Análise Exploratória**: Gera estatísticas descritivas e análises de dados categóricos
6. **Exportação**: Salva os datasets processados em formato CSV para uso em análises subsequentes


In [35]:
#from google.colab import drive
#drive.mount('/content/drive')

In [36]:
#%cd drive/MyDrive/Projeto\ disciplina\ PCS5787
#!pwd

In [37]:
import pandas as pd
import chardet
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [38]:
years = range(2019, 2025)

dfs = []
selected_columns = ["NUMERO DO CONTRIBUINTE",
                    "ANO DO EXERCICIO",
                    "NOME DE LOGRADOURO DO IMOVEL",
                    "NUMERO DO IMOVEL",
                    "AREA DO TERRENO",
                    "AREA CONSTRUIDA",
                    "AREA OCUPADA",
                    "VALOR DO M2 DO TERRENO",
                    "VALOR DO M2 DE CONSTRUCAO",
                    "TIPO DE USO DO IMOVEL",
                    "TIPO DE PADRAO DA CONSTRUCAO",
                    "TESTADA PARA CALCULO",
                    "FRACAO IDEAL",
                    "FATOR DE OBSOLESCENCIA",
                    "TIPO DE TERRENO",
                    "QUANTIDADE DE ESQUINAS FRENTES"
                    ]

for year in years:
    file_path = f"data/IPTU_{year}.csv"
    try:
        if year < 2022:
            df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig', low_memory=False)
        else:
            df = pd.read_csv(file_path, sep=';', encoding='latin-1', low_memory=False)
        if "ï»¿NUMERO DO CONTRIBUINTE" in df.columns:
            df = df.rename(columns={"ï»¿NUMERO DO CONTRIBUINTE": "NUMERO DO CONTRIBUINTE"})
        if "QUANTIDADE DE ESQUINAS/FRENTES" in df.columns:
            df = df.rename(columns={"QUANTIDADE DE ESQUINAS/FRENTES": "QUANTIDADE DE ESQUINAS FRENTES"})

        df = df[selected_columns]
        if df['VALOR DO M2 DO TERRENO'].dtype == "object":
            df['VALOR DO M2 DO TERRENO'] = df['VALOR DO M2 DO TERRENO'].str.replace(',', '.')
        if df['VALOR DO M2 DE CONSTRUCAO'].dtype == "object":
            df['VALOR DO M2 DE CONSTRUCAO'] = df['VALOR DO M2 DE CONSTRUCAO'].str.replace(',', '.')
        if df['FATOR DE OBSOLESCENCIA'].dtype == "object":
            df['FATOR DE OBSOLESCENCIA'] = df['FATOR DE OBSOLESCENCIA'].str.replace(',', '.')
        if df['TESTADA PARA CALCULO'].dtype == "object":
            df['TESTADA PARA CALCULO'] = df['TESTADA PARA CALCULO'].str.replace(',', '.')
        if df['FRACAO IDEAL'].dtype == "object":
            df['FRACAO IDEAL'] = df['FRACAO IDEAL'].str.replace(',', '.')

        dfs.append(df)
        print(f"Successfully read {file_path}")
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")



Successfully read data/IPTU_2019.csv
Successfully read data/IPTU_2020.csv
Successfully read data/IPTU_2021.csv
Successfully read data/IPTU_2022.csv
Successfully read data/IPTU_2023.csv
Successfully read data/IPTU_2024.csv


In [39]:
if dfs:
    df_combined = pd.concat(dfs, ignore_index=True)
    print("\nCombined DataFrame:")
    display(df_combined.head())
    display(df_combined.info())
else:
    print("No dataframes were loaded.")

del dfs


Combined DataFrame:


,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,AREA DO TERRENO,AREA CONSTRUIDA,AREA OCUPADA,VALOR DO M2 DO TERRENO,VALOR DO M2 DE CONSTRUCAO,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,TESTADA PARA CALCULO,FRACAO IDEAL,FATOR DE OBSOLESCENCIA,TIPO DE TERRENO,QUANTIDADE DE ESQUINAS FRENTES
0,0010030001-4,2019,R S CAETANO,13.00,136,135.00,108.00,2241.00,1668.00,Loja,Comercial horizontal - padrão B,13.00,1.0000,0.20,De esquina,1.00
1,0010030002-2,2019,R S CAETANO,19.00,90,67.00,67.00,2241.00,1668.00,Loja,Comercial horizontal - padrão B,6.00,1.0000,0.20,Normal,0.00
2,0010030003-0,2019,R S CAETANO,27.00,105,140.00,84.00,2241.00,1668.00,Loja,Comercial horizontal - padrão B,7.85,1.0000,0.32,Normal,0.00
3,0010030004-9,2019,R S CAETANO,33.00,108,103.00,86.00,2241.00,1668.00,Loja,Comercial horizontal - padrão B,6.05,1.0000,0.20,Normal,0.00
4,0010030005-7,2019,R S CAETANO,39.00,120,98.00,96.00,2241.00,1668.00,Loja,Comercial horizontal - padrão B,6.70,1.0000,0.20,Normal,0.00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21317317 entries, 0 to 21317316
Data columns (total 16 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   NUMERO DO CONTRIBUINTE          object 
 1   ANO DO EXERCICIO                int64  
 2   NOME DE LOGRADOURO DO IMOVEL    object 
 3   NUMERO DO IMOVEL                float64
 4   AREA DO TERRENO                 int64  
 5   AREA CONSTRUIDA                 float64
 6   AREA OCUPADA                    float64
 7   VALOR DO M2 DO TERRENO          object 
 8   VALOR DO M2 DE CONSTRUCAO       object 
 9   TIPO DE USO DO IMOVEL           object 
 10  TIPO DE PADRAO DA CONSTRUCAO    object 
 11  TESTADA PARA CALCULO            object 
 12  FRACAO IDEAL                    object 
 13  FATOR DE OBSOLESCENCIA          object 
 14  TIPO DE TERRENO                 object 
 15  QUANTIDADE DE ESQUINAS FRENTES  float64
dtypes: float64(4), int64(2), object(10)
memory usage: 2.5+ GB


None

In [40]:
fator_prof_map = {
    0: 0.7071, 1: 0.7071, 2: 0.7071, 3: 0.7071, 4: 0.7071,
    5: 0.7071, 6: 0.7071, 7: 0.7071, 8: 0.7071, 9: 0.7071,
    10: 0.7071, 11: 0.7416, 12: 0.7746, 13: 0.8062, 14: 0.8367,
    15: 0.8660, 16: 0.8944, 17: 0.9220, 18: 0.9487, 19: 0.9747,
    20: 1.0000, 21: 1.0000, 22: 1.0000, 23: 1.0000, 24: 1.0000,
    25: 1.0000, 26: 1.0000, 27: 1.0000, 28: 1.0000, 29: 1.0000,
    30: 1.0000, 31: 1.0000, 32: 1.0000, 33: 1.0000, 34: 1.0000,
    35: 1.0000, 36: 1.0000, 37: 1.0000, 38: 1.0000, 39: 1.0000,
    40: 1.0000, 41: 0.9877, 42: 0.9759, 43: 0.9645, 44: 0.9535,
    45: 0.9428, 46: 0.9325, 47: 0.9225, 48: 0.9129, 49: 0.9035,
    50: 0.8944, 51: 0.8856, 52: 0.8771, 53: 0.8687, 54: 0.8607,
    55: 0.8528, 56: 0.8452, 57: 0.8377, 58: 0.8305, 59: 0.8234,
    60: 0.8165, 61: 0.8098, 62: 0.8032, 63: 0.7968, 64: 0.7906,
    65: 0.7845, 66: 0.7785, 67: 0.7727, 68: 0.7670, 69: 0.7614,
    70: 0.7559, 71: 0.7506, 72: 0.7454, 73: 0.7402, 74: 0.7352,
    75: 0.7303, 76: 0.7255, 77: 0.7207, 78: 0.7161, 79: 0.7116,
    80: 0.7071, 81: 0.6984, 82: 0.6984, 83: 0.6901, 84: 0.6901,
    85: 0.6820, 86: 0.6820, 87: 0.6742, 88: 0.6742, 89: 0.6667,
    90: 0.6667, 91: 0.6594, 92: 0.6594, 93: 0.6523, 94: 0.6523,
    95: 0.6455, 96: 0.6455, 97: 0.6389, 98: 0.6389, 99: 0.6325,
    100: 0.6325, 101: 0.6172, 102: 0.6172, 103: 0.6172, 104: 0.6172,
    105: 0.6172, 106: 0.6030, 107: 0.6030, 108: 0.6030, 109: 0.6030,
    110: 0.6030, 111: 0.5898, 112: 0.5898, 113: 0.5898, 114: 0.5898,
    115: 0.5898, 116: 0.5774, 117: 0.5774, 118: 0.5774, 119: 0.5774,
    120: 0.5774, 121: 0.5657, 122: 0.5657, 123: 0.5657, 124: 0.5657,
    125: 0.5657, 126: 0.5547, 127: 0.5547, 128: 0.5547, 129: 0.5547,
    130: 0.5547, 131: 0.5443, 132: 0.5443, 133: 0.5443, 134: 0.5443,
    135: 0.5443, 136: 0.5345, 137: 0.5345, 138: 0.5345, 139: 0.5345,
    140: 0.5345, 141: 0.5252, 142: 0.5252, 143: 0.5252, 144: 0.5252,
    145: 0.5252, 146: 0.5164, 147: 0.5164, 148: 0.5164, 149: 0.5164,
    150: 0.5164, 151: 0.5000, 152: 0.5000, 153: 0.5000, 154: 0.5000,
    155: 0.5000, 156: 0.5000, 157: 0.5000, 158: 0.5000, 159: 0.5000,
    160: 0.5000, 161: 0.4851, 162: 0.4851, 163: 0.4851, 164: 0.4851,
    165: 0.4851, 166: 0.4851, 167: 0.4851, 168: 0.4851, 169: 0.4851,
    170: 0.4851, 171: 0.4714, 172: 0.4714, 173: 0.4714, 174: 0.4714,
    175: 0.4714, 176: 0.4714, 177: 0.4714, 178: 0.4714, 179: 0.4714,
    180: 0.4714, 181: 0.4588, 182: 0.4588, 183: 0.4588, 184: 0.4588,
    185: 0.4588, 186: 0.4588, 187: 0.4588, 188: 0.4588, 189: 0.4588,
    190: 0.4588, 191: 0.4472, 192: 0.4472, 193: 0.4472, 194: 0.4472,
    195: 0.4472, 196: 0.4472, 197: 0.4472, 198: 0.4472, 199: 0.4472,
    200: 0.4472
}

fator_terreno = {
    'Lote de fundos': 0.6,
    'Terreno interno': 0.7,
}

fator_ZF = {
    1: 1.3,
    2: 1.2,
    3: 1.1,
    4: 1.1
}

fator_ea = {
    1: 3,
    2: 5,
    3: 10,
    4: 10
}

area_map = {
    1: 900,
    2: 1800,
    3: 2700,
    4: 3600
}

zf_map = {
    "001": 1, "002": 1, "003": 1, "004": 2, "005": 1, "006": 1, "007": 1, "008": 1,
    "009": 1, "010": 1, "011": 1, "012": 1, "013": 1, "014": 1, "015": 1, "016": 1,
    "017": 2, "018": 2, "019": 2, "020": 1, "021": 1, "022": 1, "023": 1, "024": 1,
    "025": 1, "026": 2, "027": 2, "028": 2, "029": 2, "030": 1, "031": 2, "032": 2,
    "033": 1, "034": 1, "035": 1, "036": 1, "037": 1, "038": 1, "039": 1, "040": 1,
    "041": 1, "042": 1, "043": 2, "044": 2, "045": 1, "046": 1, "047": 1, "048": 2,
    "049": 2, "050": 2, "051": 2, "052": 2, "053": 2, "054": 1, "055": 2, "056": 2,
    "057": 2, "058": 2, "059": 2, "060": 2, "061": 2, "062": 2, "063": 3, "064": 2,
    "065": 2, "066": 2, "067": 2, "068": 2, "069": 1, "070": 2, "071": 2, "072": 1,
    "073": 1, "074": 2, "075": 2, "076": 2, "077": 2, "078": 2, "079": 2, "080": 1,
    "081": 1, "082": 2, "083": 1, "084": 1, "085": 1, "086": 1, "087": 2, "088": 2,
    "089": 2, "090": 2, "091": 2, "093": 2, "094": 3, "095": 2, "096": 1, "097": 2,
    "098": 2, "099": 2, "100": 2, "101": 2, "102": 2, "103": 2, "104": 2, "105": 2,
    "106": 3, "107": 2, "108": 2, "109": 2, "110": 2, "111": 3, "112": 3, "113": 2,
    "114": 3, "115": 3, "116": 2, "117": 2, "118": 2, "119": 2, "120": 2, "121": 2,
    "122": 2, "123": 2, "124": 3, "125": 2, "126": 3, "127": 3, "128": 3, "129": 3,
    "130": 3, "131": 3, "132": 3, "133": 3, "134": 3, "135": 3, "136": 3, "137": 3,
    "138": 3, "139": 3, "140": 3, "141": 3, "142": 3, "143": 3, "144": 3, "145": 3,
    "146": 3, "147": 3, "148": 2, "149": 3, "150": 3, "151": 3, "152": 3, "153": 3,
    "154": 3, "155": 3, "156": 3, "157": 3, "158": 3, "159": 2, "160": 2, "161": 3,
    "162": 3, "163": 2, "164": 3, "165": 3, "166": 3, "167": 3, "168": 3, "169": 3,
    "170": 2, "171": 2, "172": 3, "173": 3, "174": 3, "175": 3, "176": 3, "177": 3,
    "178": 3, "179": 3, "180": 3, "181": 3, "182": 3, "183": 3, "184": 3, "185": 3,
    "186": 3, "187": 3, "188": 3, "189": 3, "190": 3, "191": 3, "192": 3, "193": 3,
    "194": 3, "195": 3, "196": 2, "197": 2, "198": 3, "199": 2, "200": 1, "201": 4,
    "202": 4, "203": 4, "204": 4, "205": 4, "206": 4, "207": 4, "208": 4, "209": 3,
    "210": 4, "211": 4, "212": 3, "213": 4, "214": 3, "215": 4, "216": 4, "217": 4,
    "218": 4, "219": 4, "220": 4, "221": 4, "222": 4, "223": 4, "224": 4, "225": 4,
    "226": 4, "227": 4, "228": 4, "229": 3, "230": 3, "231": 4, "232": 4, "233": 3,
    "234": 3, "235": 3, "236": 4, "237": 3, "238": 4, "239": 4, "240": 3, "241": 4,
    "242": 3, "243": 4, "244": 4, "245": 4, "246": 4, "247": 4, "248": 4, "249": 4,
    "250": 4, "251": 4, "252": 3, "253": 3, "254": 4, "255": 4, "256": 4, "257": 4,
    "258": 4, "259": 4, "260": 4, "261": 4, "262": 4, "263": 4, "264": 4, "265": 4,
    "266": 4, "267": 4, "268": 4, "269": 4, "270": 4, "271": 4, "272": 4, "273": 4,
    "274": 4, "275": 4, "276": 4, "277": 4, "278": 4, "279": 4, "280": 4, "281": 4,
    "282": 4, "283": 4, "284": 4, "285": 4, "286": 4, "287": 4, "288": 4, "289": 4,
    "290": 4, "291": 4, "292": 4, "293": 4, "294": 4, "295": 4, "296": 4, "297": 4,
    "298": 4, "299": 1, "300": 1, "301": 2, "302": 2, "303": 2, "304": 2, "305": 2,
    "306": 2, "307": 3, "308": 3, "309": 1, "310": 2
}


In [41]:
def calcula_vv(row):
  vv_c = float(row['AREA CONSTRUIDA'])*float(row['VALOR DO M2 DE CONSTRUCAO'])*float(row['FATOR DE OBSOLESCENCIA'])

  area_incorporada = float(row['AREA DO TERRENO'])
  excesso_area = 0
  if float(row['AREA OCUPADA']) < row['AREA DO TERRENO']/fator_ea[row['ZF']]:
    area_incorporada = (fator_ea[row['ZF']]+1)*row['AREA OCUPADA']
    excesso_area = row['AREA DO TERRENO'] - area_incorporada

  fator_prof = 1
  if float(row['TESTADA PARA CALCULO']) > 0.1:
    fator_prof = fator_prof_map[min(int(float(row['AREA DO TERRENO'])/float(row['TESTADA PARA CALCULO'])),200)]

  vv_t = area_incorporada*float(row['VALOR DO M2 DO TERRENO'])*fator_prof
  vv_e = excesso_area*float(row['VALOR DO M2 DO TERRENO'])*fator_prof

  if row['TIPO DE TERRENO'] in fator_terreno:
    vv_t = vv_t*fator_terreno[row['TIPO DE TERRENO']]
    vv_e = vv_e*fator_terreno[row['TIPO DE TERRENO']]

  fator_esq = 1
  area_esq = 0
  if row['TIPO DE TERRENO'] == 'De esquina' or row['TIPO DE TERRENO'] == 'Lote de esquina em ZER':
    area_esq = min(area_map[min(row['QUANTIDADE DE ESQUINAS FRENTES'],4)], row['AREA DO TERRENO'])
    fator_esq = 1+(fator_ZF[row['ZF']]-1)*area_esq/row['AREA DO TERRENO']

  vv_t = vv_t*fator_esq
  vv_e = vv_e*fator_esq

  fator_condo = 1

  if row['FRACAO IDEAL'] != 1 and (row['TIPO DE PADRAO DA CONSTRUCAO'].startswith('Residencial vertical') or row['TIPO DE PADRAO DA CONSTRUCAO'].startswith('Comercial vertical')) and row['TIPO DE USO DO IMOVEL'] != 'TERRENO':
      teste_condo = vv_t*float(row['FRACAO IDEAL'])*1.6/vv_c
      if teste_condo < 0.2:
          fator_condo = (2.2-3*teste_condo)
      elif teste_condo < 2.01:
          fator_condo = 1.6
      elif teste_condo < 7:
          fator_condo = 1.8-teste_condo/10
      else:
          fator_condo = 1.1
      vv_t = vv_t*fator_condo
      vv_e = vv_e*fator_condo

  vv_t = vv_t*float(row['FRACAO IDEAL'])
  vv_e = vv_e*float(row['FRACAO IDEAL'])

  # print(fator_condo)
  # print(fator_prof)
  # print(fator_esq)
  # print(vv_c)
  # print(vv_t)
  # print(vv_e)
  return vv_c+vv_t+vv_e

In [42]:
df_combined['ZF'] = df_combined.apply(lambda row: zf_map[row['NUMERO DO CONTRIBUINTE'].zfill(11)[:3]], axis=1)

In [43]:
df_combined = df_combined[pd.to_numeric(df_combined['TESTADA PARA CALCULO'], errors='coerce').notna()]

In [44]:
data_types = {
    "NUMERO DO CONTRIBUINTE": "string",
    "ANO DO EXERCICIO": "string",
    "NOME DE LOGRADOURO DO IMOVEL": "string",
    "NUMERO DO IMOVEL": "Int64",
    "AREA DO TERRENO": "float64",
    "VALOR DO M2 DO TERRENO": "Int64",
    "VALOR DO M2 DE CONSTRUCAO": "Int64",
    "TIPO DE USO DO IMOVEL": "string",
    "TIPO DE PADRAO DA CONSTRUCAO": "string",
    "TESTADA PARA CALCULO": "float64",
    "FRACAO IDEAL": "float64",
    "FATOR DE OBSOLESCENCIA": "float64",
    "QUANTIDADE DE ESQUINAS FRENTES": "Int64",
    "TIPO DE TERRENO": "string",
}

for col, dtype in data_types.items():
    # Handle potential errors during conversion
    try:
        if dtype == 'string':
            df_combined[col] = df_combined[col].astype(str)
        elif dtype == 'Int64':
            df_combined[col] = pd.to_numeric(df_combined[col], errors='coerce').astype('Int64')
        else:
            df_combined[col] = df_combined[col].astype(dtype)
        print(f"Successfully converted {col} to {dtype}")
    except Exception as e:
        print(f"Error converting {col} to {dtype}: {e}")

df_combined['NUMERO DO CONTRIBUINTE'] = df_combined['NUMERO DO CONTRIBUINTE'].str.replace(r'[^A-Za-z0-9 ]', '', regex=True)
display(df_combined.info())
display(df_combined.head())

Successfully converted NUMERO DO CONTRIBUINTE to string
Successfully converted ANO DO EXERCICIO to string
Successfully converted NOME DE LOGRADOURO DO IMOVEL to string
Successfully converted NUMERO DO IMOVEL to Int64
Successfully converted AREA DO TERRENO to float64
Successfully converted VALOR DO M2 DO TERRENO to Int64
Successfully converted VALOR DO M2 DE CONSTRUCAO to Int64
Successfully converted TIPO DE USO DO IMOVEL to string
Successfully converted TIPO DE PADRAO DA CONSTRUCAO to string
Successfully converted TESTADA PARA CALCULO to float64
Successfully converted FRACAO IDEAL to float64
Successfully converted FATOR DE OBSOLESCENCIA to float64
Successfully converted QUANTIDADE DE ESQUINAS FRENTES to Int64
Successfully converted TIPO DE TERRENO to string
<class 'pandas.core.frame.DataFrame'>
Index: 21317311 entries, 0 to 21317316
Data columns (total 17 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   NUMERO DO CONTRIBUINTE     

None

,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,AREA DO TERRENO,AREA CONSTRUIDA,AREA OCUPADA,VALOR DO M2 DO TERRENO,VALOR DO M2 DE CONSTRUCAO,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,TESTADA PARA CALCULO,FRACAO IDEAL,FATOR DE OBSOLESCENCIA,TIPO DE TERRENO,QUANTIDADE DE ESQUINAS FRENTES,ZF
0,00100300014,2019,R S CAETANO,13,136.00,135.00,108.00,2241,1668,Loja,Comercial horizontal - padrão B,13.00,1.00,0.20,De esquina,1,1
1,00100300022,2019,R S CAETANO,19,90.00,67.00,67.00,2241,1668,Loja,Comercial horizontal - padrão B,6.00,1.00,0.20,Normal,0,1
2,00100300030,2019,R S CAETANO,27,105.00,140.00,84.00,2241,1668,Loja,Comercial horizontal - padrão B,7.85,1.00,0.32,Normal,0,1
3,00100300049,2019,R S CAETANO,33,108.00,103.00,86.00,2241,1668,Loja,Comercial horizontal - padrão B,6.05,1.00,0.20,Normal,0,1
4,00100300057,2019,R S CAETANO,39,120.00,98.00,96.00,2241,1668,Loja,Comercial horizontal - padrão B,6.70,1.00,0.20,Normal,0,1


In [45]:
df_combined['VALOR VENAL'] = df_combined.apply(lambda row: calcula_vv(row), axis=1)

In [46]:
selected_columns = ["NUMERO DO CONTRIBUINTE",
                    "ANO DO EXERCICIO",
                    "NOME DE LOGRADOURO DO IMOVEL",
                    "NUMERO DO IMOVEL",
                    "AREA DO TERRENO",
                    "AREA CONSTRUIDA",
                    "AREA OCUPADA",
                    "VALOR DO M2 DO TERRENO",
                    "VALOR DO M2 DE CONSTRUCAO",
                    "TIPO DE USO DO IMOVEL",
                    "TIPO DE PADRAO DA CONSTRUCAO",
                    "VALOR VENAL"
                    ]
df_combined = df_combined[selected_columns]

In [47]:
df_combined.to_csv('data/IPTU_2019_2024.csv', index=False)

## Generate descriptive statistics


In [48]:

numerical_columns = df_combined.select_dtypes(include=['int64', 'float64'])
descriptive_stats = numerical_columns.describe()
display(descriptive_stats)

,NUMERO DO IMOVEL,AREA DO TERRENO,AREA CONSTRUIDA,AREA OCUPADA,VALOR DO M2 DO TERRENO,VALOR DO M2 DE CONSTRUCAO,VALOR VENAL
count,20804861.00,21317311.00,21317311.00,21317311.00,21317311.00,21317311.00,21317311.00
mean,581.87,3986.79,154.29,1304.12,2462.21,1846.71,437341.05
std,1124.40,16923.68,1057.46,3451.62,3144.23,819.97,4474126.24
min,0.00,1.00,0.00,0.00,45.00,0.00,78.01
25%,102.00,185.00,63.00,90.00,682.00,1322.00,112866.61
50%,262.00,711.00,100.00,324.00,1331.00,1774.00,209142.38
75%,619.00,2481.00,153.00,1234.00,2870.00,2291.00,379048.16
max,99541.00,4307493.00,950328.00,464200.00,32871.00,4249.00,4861333058.59


## Analyze categorical data


In [49]:
categorical_columns = df_combined.select_dtypes(include='object').columns.tolist()

categorical_columns.remove('NUMERO DO CONTRIBUINTE')
categorical_columns.remove('ANO DO EXERCICIO')
#categorical_columns.remove('NUMERO DA NL')
#categorical_columns.remove('CODLOG DO IMOVEL')

for col in categorical_columns:
    print(f"\nValue counts for column: {col}")
    # Display top 20 categories for columns with many unique values
    if df_combined[col].nunique() > 20:
        display(df_combined[col].value_counts().head(20))
    else:
        display(df_combined[col].value_counts())


Value counts for column: NOME DE LOGRADOURO DO IMOVEL


NOME DE LOGRADOURO DO IMOVEL
AV  PAULISTA                         77929
AV  RAIMUNDO PEREIRA DE MAGALHAES    55815
R DA CONSOLACAO                      48862
AV BRIG LUIS ANTONIO                 46957
R  VERGUEIRO                         46871
AV  NOVE DE JULHO                    43988
AV BRIG FARIA LIMA                   43365
AV  CELSO GARCIA                     40689
R  BELA CINTRA                       37504
R  VOLUNTARIOS DA PATRIA             37337
AV  ANGELICA                         36800
AV  INTERLAGOS                       35613
R FREI CANECA                        33456
AL  SANTOS                           31434
AL  LORENA                           29945
AV DAS NACOES UNIDAS                 28548
AV PDE ARLINDO VIEIRA                28472
AV  VILA EMA                         27459
R  AUGUSTA                           26127
R  OSCAR FREIRE                      25828
Name: count, dtype: int64


Value counts for column: TIPO DE USO DO IMOVEL


TIPO DE USO DO IMOVEL
Apartamento em condomínio                                                                     8521073
Residência                                                                                    6491698
Residência coletiva, exclusive cortiço (mais de uma residência no lote)                       1225981
Garagem (unidade autônoma) em edifício em condomínio de uso exclusivamente residencial        1185071
Escritório/consultório em condomínio (unidade autônoma)                                        831480
Terreno                                                                                        628525
Residência e outro uso (predominância residencial)                                             512335
Loja                                                                                           324601
Loja e residência (predominância comercial)                                                    266636
Garagem (unidade autônoma) em edifício em condomínio de escr


Value counts for column: TIPO DE PADRAO DA CONSTRUCAO


TIPO DE PADRAO DA CONSTRUCAO
Residencial vertical - padrão C                                                     6382194
Residencial horizontal - padrão C                                                   3914083
Residencial horizontal - padrão B                                                   2366972
Residencial horizontal - padrão A                                                   1712458
Residencial vertical - padrão D                                                     1562978
Residencial vertical - padrão B                                                      977441
Comercial vertical - padrão C                                                        697734
TERRENO                                                                              628525
Comercial horizontal - padrão B                                                      546644
Residencial vertical - padrão A                                                      504219
Comercial vertical - padrão B                      

In [50]:
years = range(2019, 2025)
months = ['JAN', 'FEV', 'MAR', 'ABR', 'MAI', 'JUN',
          'JUL', 'AGO', 'SET', 'OUT', 'NOV', 'DEZ']

dfs_itbi = []

selected_columns = ["N° do Cadastro (SQL)",
                    "Natureza de Transação",
                    "Valor de Transação (declarado pelo contribuinte)",
                    "Data de Transação",
                    "Valor Venal de Referência",
                    "Proporção Transmitida (%)",
                    "Valor Venal de Referência (proporcional)",
                    "Base de Cálculo adotada",
                    "Tipo de Financiamento",
                    "Valor Financiado",
                    "Situação do SQL"
                    ]

for year in years:
    file_path = f"data/xlsx/ITBI_{year}.xlsx"
    df = None
    for month in months:
      sheet_name = f"{month}-{year}"
      try:
          df = pd.read_excel(file_path, sheet_name=sheet_name)
          df = df[selected_columns]
          print(f"Successfully read {file_path} {sheet_name}")
      except FileNotFoundError:
          print(f"File not found: {file_path}, {sheet_name}")
      except Exception as e:
          print(f"Error reading {file_path}, {sheet_name}: {e}")
      dfs_itbi.append(df)



Successfully read data/xlsx/ITBI_2019.xlsx JAN-2019
Successfully read data/xlsx/ITBI_2019.xlsx FEV-2019
Successfully read data/xlsx/ITBI_2019.xlsx MAR-2019
Successfully read data/xlsx/ITBI_2019.xlsx ABR-2019
Successfully read data/xlsx/ITBI_2019.xlsx MAI-2019
Successfully read data/xlsx/ITBI_2019.xlsx JUN-2019
Successfully read data/xlsx/ITBI_2019.xlsx JUL-2019
Successfully read data/xlsx/ITBI_2019.xlsx AGO-2019
Successfully read data/xlsx/ITBI_2019.xlsx SET-2019
Successfully read data/xlsx/ITBI_2019.xlsx OUT-2019
Successfully read data/xlsx/ITBI_2019.xlsx NOV-2019
Successfully read data/xlsx/ITBI_2019.xlsx DEZ-2019
Successfully read data/xlsx/ITBI_2020.xlsx JAN-2020
Successfully read data/xlsx/ITBI_2020.xlsx FEV-2020
Successfully read data/xlsx/ITBI_2020.xlsx MAR-2020
Successfully read data/xlsx/ITBI_2020.xlsx ABR-2020
Successfully read data/xlsx/ITBI_2020.xlsx MAI-2020
Successfully read data/xlsx/ITBI_2020.xlsx JUN-2020
Successfully read data/xlsx/ITBI_2020.xlsx JUL-2020
Successfully

In [51]:
if dfs_itbi:
    df_combined_itbi = pd.concat(dfs_itbi, ignore_index=True)
    print("\nCombined DataFrame:")
    display(df_combined_itbi.head())
    display(df_combined_itbi.info())
else:
    print("No dataframes were loaded.")


Combined DataFrame:


,N° do Cadastro (SQL),Natureza de Transação,Valor de Transação (declarado pelo contribuinte),Data de Transação,Valor Venal de Referência,Proporção Transmitida (%),Valor Venal de Referência (proporcional),Base de Cálculo adotada,Tipo de Financiamento,Valor Financiado,...,173753,Ativo Territorial,18467,47.63,0.1,0.2,TERRENO,0.3,TERRENO.1,Unnamed: 27
0,8010602450,1.Compra e venda,600000.00,2018-12-19 00:00:00,518060.00,100.00,518060.00,600000.00,1.Sistema Financeiro de Habitação,300000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16900918084,1.Compra e venda,211000.00,2018-12-21 00:00:00,208709.00,100.00,208709.00,211000.00,1.Sistema Financeiro de Habitação,155000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13611100195,1.Compra e venda,180000.00,2018-12-20 00:00:00,426634.00,100.00,426634.00,426634.00,1.Sistema Financeiro de Habitação,80000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4111100708,1.Compra e venda,910000.00,2018-12-21 00:00:00,547964.00,100.00,547964.00,910000.00,1.Sistema Financeiro de Habitação,250000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4706402441,1.Compra e venda,857142.86,2018-12-20 00:00:00,909231.00,100.00,909231.00,909231.00,1.Sistema Financeiro de Habitação,400000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947162 entries, 0 to 947161
Data columns (total 62 columns):
 #   Column                                            Non-Null Count   Dtype  
---  ------                                            --------------   -----  
 0   N° do Cadastro (SQL)                              913982 non-null  object 
 1   Natureza de Transação                             913982 non-null  object 
 2   Valor de Transação (declarado pelo contribuinte)  913982 non-null  object 
 3   Data de Transação                                 913982 non-null  object 
 4   Valor Venal de Referência                         913982 non-null  float64
 5   Proporção Transmitida (%)                         913982 non-null  float64
 6   Valor Venal de Referência (proporcional)          913982 non-null  float64
 7   Base de Cálculo adotada                           913608 non-null  float64
 8   Tipo de Financiamento                             351905 non-null  object 
 9   Valo

None

In [52]:
data_types = {
    "N° do Cadastro (SQL)": "string",
    "Natureza de Transação": "string",
    "Valor de Transação (declarado pelo contribuinte)": "float64",
    "Valor Venal de Referência": "float64",
    "Proporção Transmitida (%)": "float64",
    "Valor Venal de Referência (proporcional)": "float64",
    "Base de Cálculo adotada": "float64",
    "Valor Financiado": "float64",
    "Situação do SQL": "string"
}

for col, dtype in data_types.items():
    try:
        if dtype == 'string':
            df_combined_itbi[col] = df_combined_itbi[col].astype(str)
        elif dtype == 'Int64':
            df_combined_itbi[col] = pd.to_numeric(df_combined_itbi[col], errors='coerce').astype('Int64')
        else:
            df_combined_itbi[col] = df_combined_itbi[col].astype(dtype)
        print(f"Successfully converted {col} to {dtype}")
    except Exception as e:
        print(f"Error converting {col} to {dtype}: {e}")

display(df_combined_itbi.info())
display(df_combined_itbi.head())

Successfully converted N° do Cadastro (SQL) to string
Successfully converted Natureza de Transação to string
Error converting Valor de Transação (declarado pelo contribuinte) to float64: could not convert string to float: 'HELENO LOURENCO DA SILVA'
Successfully converted Valor Venal de Referência to float64
Successfully converted Proporção Transmitida (%) to float64
Successfully converted Valor Venal de Referência (proporcional) to float64
Successfully converted Base de Cálculo adotada to float64
Successfully converted Valor Financiado to float64
Successfully converted Situação do SQL to string
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947162 entries, 0 to 947161
Data columns (total 62 columns):
 #   Column                                            Non-Null Count   Dtype  
---  ------                                            --------------   -----  
 0   N° do Cadastro (SQL)                              947162 non-null  object 
 1   Natureza de Transação                     

None

,N° do Cadastro (SQL),Natureza de Transação,Valor de Transação (declarado pelo contribuinte),Data de Transação,Valor Venal de Referência,Proporção Transmitida (%),Valor Venal de Referência (proporcional),Base de Cálculo adotada,Tipo de Financiamento,Valor Financiado,...,173753,Ativo Territorial,18467,47.63,0.1,0.2,TERRENO,0.3,TERRENO.1,Unnamed: 27
0,8010602450,1.Compra e venda,600000.00,2018-12-19 00:00:00,518060.00,100.00,518060.00,600000.00,1.Sistema Financeiro de Habitação,300000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16900918084,1.Compra e venda,211000.00,2018-12-21 00:00:00,208709.00,100.00,208709.00,211000.00,1.Sistema Financeiro de Habitação,155000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13611100195,1.Compra e venda,180000.00,2018-12-20 00:00:00,426634.00,100.00,426634.00,426634.00,1.Sistema Financeiro de Habitação,80000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4111100708,1.Compra e venda,910000.00,2018-12-21 00:00:00,547964.00,100.00,547964.00,910000.00,1.Sistema Financeiro de Habitação,250000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4706402441,1.Compra e venda,857142.86,2018-12-20 00:00:00,909231.00,100.00,909231.00,909231.00,1.Sistema Financeiro de Habitação,400000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
df_combined_itbi['N° do Cadastro (SQL)'] = df_combined_itbi['N° do Cadastro (SQL)'].str.zfill(11)

df_combined_itbi['Data de Transação'] = pd.to_datetime(df_combined_itbi['Data de Transação'], errors='coerce')
df_combined_itbi = df_combined_itbi[df_combined_itbi['Data de Transação'].dt.year.between(2019, 2024)]
df_combined_itbi['Ano da Transacao'] = df_combined_itbi['Data de Transação'].dt.year
df_combined_itbi['Mes da Transacao'] = df_combined_itbi['Data de Transação'].dt.month
df_combined_itbi['Ano/Mes da Transacao'] = df_combined_itbi['Data de Transação'].dt.year.astype(str)+'/'+df_combined_itbi['Data de Transação'].dt.month.astype(str).str.zfill(2)
display(df_combined_itbi.head(20))

,N° do Cadastro (SQL),Natureza de Transação,Valor de Transação (declarado pelo contribuinte),Data de Transação,Valor Venal de Referência,Proporção Transmitida (%),Valor Venal de Referência (proporcional),Base de Cálculo adotada,Tipo de Financiamento,Valor Financiado,...,47.63,0.1,0.2,TERRENO,0.3,TERRENO.1,Unnamed: 27,Ano da Transacao,Mes da Transacao,Ano/Mes da Transacao
127,12310100659,1.Compra e venda,900000.00,2019-01-02,773836.00,100.00,773836.00,900000.00,1.Sistema Financeiro de Habitação,400000.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
133,05528502403,1.Compra e venda,420000.00,2019-01-02,303832.00,100.00,303832.00,420000.00,1.Sistema Financeiro de Habitação,340721.07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
147,29903601032,1.Compra e venda,650000.00,2019-01-03,459364.00,100.00,459364.00,650000.00,NaN,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
156,04201104559,1.Compra e venda,61765.36,2019-01-03,300840.00,100.00,300840.00,300840.00,NaN,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
157,02206602970,1.Compra e venda,965000.00,2019-01-03,575501.00,100.00,575501.00,965000.00,NaN,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
159,04201104559,1.Compra e venda,62092.62,2019-01-03,300840.00,100.00,300840.00,300840.00,NaN,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
163,01500803995,1.Compra e venda,690000.00,2019-01-03,351710.00,100.00,351710.00,690000.00,NaN,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
167,23007502991,1.Compra e venda,175000.00,2019-01-02,47460.00,100.00,47460.00,175000.00,1.Sistema Financeiro de Habitação,138835.37,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
174,09016400885,1.Compra e venda,380000.00,2019-01-02,283295.00,100.00,283295.00,380000.00,NaN,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01
175,09045201674,1.Compra e venda,440000.00,2019-01-03,453153.00,100.00,453153.00,453153.00,NaN,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,2019/01


In [54]:
missing_values = df_combined_itbi.isnull().sum()
display(missing_values)

N° do Cadastro (SQL)                                     0
Natureza de Transação                                    0
Valor de Transação (declarado pelo contribuinte)         0
Data de Transação                                        0
Valor Venal de Referência                                0
                                                     ...  
TERRENO.1                                           908503
Unnamed: 27                                         908503
Ano da Transacao                                         0
Mes da Transacao                                         0
Ano/Mes da Transacao                                     0
Length: 65, dtype: int64

In [55]:
numerical_columns = df_combined_itbi.select_dtypes(include=['int64', 'float64'])
descriptive_stats = numerical_columns.describe()
display(descriptive_stats)

,Valor Venal de Referência,Proporção Transmitida (%),Valor Venal de Referência (proporcional),Base de Cálculo adotada,Valor Financiado
count,908503.00,908503.00,908503.00,908331.00,908503.00
mean,4454465.95,68.98,413847.90,672730.87,104144.32
std,20350340.46,44.53,2144920.41,2933203.03,183714.54
min,0.00,0.00,0.00,0.00,-604339.00
25%,230383.00,1.43,38985.82,230000.00,0.00
50%,447013.00,100.00,208365.00,330000.00,0.00
75%,1659357.00,100.00,417948.00,600000.00,188000.00
max,1652865322.00,823.64,488407868.00,729435080.10,18050000.00


In [56]:
categorical_columns = df_combined_itbi.select_dtypes(include='object').columns.tolist()
display(categorical_columns)
categorical_columns.remove('N° do Cadastro (SQL)')

for col in categorical_columns:
    print(f"\nValue counts for column: {col}")
    if df_combined_itbi[col].nunique() > 20:
        display(df_combined_itbi[col].value_counts().head(20))
    else:
        display(df_combined_itbi[col].value_counts())

['N° do Cadastro (SQL)',
 'Natureza de Transação',
 'Valor de Transação (declarado pelo contribuinte)',
 'Tipo de Financiamento',
 'Situação do SQL',
 100800297,
 'R S LAZARO',
 295,
 'Unnamed: 3',
 'Unnamed: 4',
 'Unnamed: 5',
 1103020,
 '1.Compra e venda',
 450000,
 datetime.datetime(2024, 1, 16, 0, 0),
 190531,
 100,
 '190531.1',
 '450000.1',
 'Unnamed: 14',
 0,
 '5º Cartório de Registro de Imóvel',
 42915,
 'Ativo Predial',
 72,
 5.9,
 1,
 120,
 10,
 'RESIDÊNCIA',
 12,
 'RESIDENCIAL HORIZONTAL ',
 1939,
 9803003283,
 'R FORTUNATO FERRAZ',
 99999,
 'Apto 231 - Torre 3',
 'VILA ANASTACIO',
 5093000,
 1349864.78,
 datetime.datetime(2024, 9, 27, 0, 0),
 19349531,
 0.25,
 48373.8275,
 '1349864.78.1',
 '10º Cartório de Registro de Imóvel',
 173753,
 'Ativo Territorial',
 18467,
 47.63,
 '0.1',
 '0.2',
 'TERRENO',
 '0.3',
 'TERRENO.1',
 'Unnamed: 27',
 'Ano/Mes da Transacao']


Value counts for column: Natureza de Transação


Natureza de Transação
1.Compra e venda                                                          821021
12.Dação em pagamento por escritura pública                                20283
20.Realização ou integralização de capital                                 11184
2.Cessão de direitos relativos a compromisso de compra e venda              8959
15.Permuta por escritura pública                                            8568
4.Arrematação (em leilão ou hasta pública)                                  8151
17.Resolução da alienação fiduciária por inadimplemento                     7376
33.Demais atos onerosos translativos                                        6100
3.Adjudicação (vinculada a processo de execução ou cobrança de dívida)      3178
26.Cessão de direitos sobre o imóvel com alienação fiduciária               2617
8.Cessão de direitos hereditários (ou sucessórios)                          1908
5.Adjudicação compulsória                                                   1728
27.Exc


Value counts for column: Valor de Transação (declarado pelo contribuinte)


Valor de Transação (declarado pelo contribuinte)
264000.00    17841
240000.00    17074
350000.00    14313
300000.00    13922
250000.00    11828
400000.00    10185
200000.00     9834
500000.00     8042
230000.00     6902
260000.00     6821
450000.00     6710
280000.00     6397
320000.00     6143
220000.00     6141
600000.00     6100
100000.00     5950
150000.00     5405
330000.00     5362
270000.00     5137
50000.00      4910
Name: count, dtype: int64

...

Series([], Name: count, dtype: int64)


Value counts for column: Unnamed: 27


Series([], Name: count, dtype: int64)


Value counts for column: Ano/Mes da Transacao


Ano/Mes da Transacao
2024/08    20309
2024/04    20222
2024/07    19414
2024/06    18589
2024/05    18365
2024/11    17767
2024/03    17016
2023/08    16918
2021/12    16891
2024/12    16243
2023/05    16138
2021/08    16130
2023/03    16059
2021/11    15856
2023/06    15739
2023/09    15318
2023/10    15227
2020/12    15219
2022/12    15198
2021/09    15140
Name: count, dtype: int64

In [57]:
transactions_by_month_year = df_combined_itbi.groupby('Ano/Mes da Transacao').size().reset_index(name='Numero de Transacoes')
display(transactions_by_month_year)

,Ano/Mes da Transacao,Numero de Transacoes
0,2019/01,5669
1,2019/02,7129
2,2019/03,7414
3,2019/04,8579
4,2019/05,9203
...,...,...
67,2024/08,20309
68,2024/09,15057
69,2024/10,3998
70,2024/11,17767


In [58]:
df_combined_itbi.to_csv('data/ITBI_2019_2024.csv', index=False)